# Character-Level Text Generation with CNN, LSTM, RNN, and GRU Models

This notebook presents a complete character-level text generation project using multiple deep learning
architectures. The objective is to train neural networks that learn the statistical structure of text and
generate new sequences in the style of *Alice's Adventures in Wonderland*.

## Project Goals
- Build and train several sequence models:
  - **CharCNN** (1D convolutional model)
  - **LSTM** (baseline recurrent model)
  - **Modified LSTM** with variations in:
    - number of layers
    - hidden state dimensionality
    - dropout rates
    - other relevant hyperparameters
  - **RNN** and **GRU** alternatives for comparison

- Construct a character-level dataset using sliding windows of fixed length  
- Train all models using **cross-entropy loss**  
- Evaluate performance using:
  - **Character-level accuracy**
  - **Confusion matrices**
  - **Loss curves and training dynamics**

- Generate new text sequences from each trained model  
- Analyze and compare the generative behavior of CNN, LSTM, RNN, and GRU architectures  

## Key Learning Outcomes
- Understanding how different sequence models handle long-range dependencies  
- Observing the impact of architectural choices on prediction accuracy  
- Using confusion matrices to diagnose model errors  
- Exploring how hyperparameters influence generative quality  
- Gaining practical experience with character-level language modeling  

In [1]:
# Use GPU if available
import torch
train_on_gpu = torch.cuda.is_available()
if not train_on_gpu:
    print('CUDA is not available.  Training on CPU ...')
else:
    print('CUDA is available!  Training on GPU ...')
device = torch.device("cuda:0" if train_on_gpu else "cpu")
print(device)

CUDA is not available.  Training on CPU ...
cpu


In [2]:
# mount with drive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [3]:
#change working directory
import os
os.chdir('/content/drive/MyDrive/cnn-lstm-textgen/')

In [4]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import re

# 1. Load dataset

We are going to use the text of the book "Alice's Adventures in Wonderland" to train our models. We need to download the text and prepare a dataset to load strings to train our models.


In [5]:
# Download the text file "Alice's Adventures in Wonderland" from Project Gutenberg
!wget -O wonderland.txt https://www.gutenberg.org/ebooks/11.txt.utf-8

# Load the text data from the downloaded file
filename = "wonderland.txt"
with open(filename, 'r', encoding='utf-8') as f:
  raw_text = f.read()

# Convert all characters to lowercase
raw_text = raw_text.lower()

# Remove non-alphanumeric characters
raw_text = re.sub(r'\n', ' ', raw_text)
raw_text = re.sub(r'[^A-Za-z ]+', '', raw_text)

print(raw_text[500:700])
# Create a set of unique characters in the text
unique_chars = set(raw_text)

# Sort the unique characters
chars = sorted(list(unique_chars))

# Create a dictionary mapping each unique character to a unique integer
char_to_int = dict((c, i) for i, c in enumerate(chars))

# Split the text into training and testing sets
train_start = int(len(raw_text) * 0.1)  # Starting index for training set (10% of text)
train_end = int(len(raw_text) * 0.8)  # Ending index for training set (80% of text)
test_start = train_end  # Starting index for testing set (remaining 20% of text)

raw_text_train = raw_text[train_start:train_end]  # Extract training text
raw_text_test = raw_text[test_start:]  # Extract testing text

# Calculate and print some summary statistics
n_chars_train = len(raw_text_train)
n_chars_test = len(raw_text_test)
n_vocab = len(chars)

print("Total Characters train:", n_chars_train)
print("Total Characters test:", n_chars_test)
print("Total Unique Characters (Vocabulary Size):", n_vocab)
print("Chars: ",chars)

--2026-05-07 13:21:19--  https://www.gutenberg.org/ebooks/11.txt.utf-8
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: http://www.gutenberg.org/cache/epub/11/pg11.txt [following]
--2026-05-07 13:21:20--  http://www.gutenberg.org/cache/epub/11/pg11.txt
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://www.gutenberg.org/cache/epub/11/pg11.txt [following]
--2026-05-07 13:21:20--  https://www.gutenberg.org/cache/epub/11/pg11.txt
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 174314 (170K) [text/plain]
Saving to: ‘wonderland.txt’

wonderland.txt      100%[===================>] 170.23K  --.-KB/s    in 0.1s   